In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "ebel2020object")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "ObjPlay_data_glmm.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)


df['study_id']="ebel2020object"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df.rename(columns={"subject": "participant"}, inplace=True)

In [3]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['participant'] = df['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
df= df.merge(apedf,left_on='participant', right_on='name', how='left') 


In [4]:
spe_2=[]  #group id based on species list (if species only present for wkprc participants)
for index, row in df.iterrows():
    if pd.isna(row['sex_y']):
        spe_2.append(row['sex_x'])
    else:
        spe_2.append(row['sex_y'])
df = df.assign(sex=spe_2)

df['sex'].replace('female', 'f', inplace=True)
df['sex'].replace('male', 'm', inplace=True)
# df.columns

df.rename(columns={"age": "age_in_years"}, inplace=True)

In [5]:
df['species'].unique()
df['species'].replace(np.nan, 'orangutan', inplace=True, regex=True)
df.dropna(subset=['participant'], inplace=True)

In [6]:
studyID_standardized=df[['study_id','participant', 'age_in_years','sex',  'species', 'zoo', 'condition', 'texture',
       'shape', 'colour', 'contact_time_sec', 'contact_proportion',
       'trial_time_sec', 'manipul_count', 'manipul_proportion']]

comp_out_path_stand = os.path.join(out_pathway, 'ebel2020object_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


In [7]:


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'ebel2020object_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)